In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import xml.etree.ElementTree as ET
import os
import typing
import numpy as np
SUMO_START_TIME = 28800
# Increase font sizes globally
mpl.rcParams['axes.titlesize'] = 40
mpl.rcParams['axes.labelsize'] = 20
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 15
mpl.rcParams['legend.fontsize'] = 30
mpl.rcParams['figure.titlesize'] = 30
# line width
mpl.rcParams['lines.linewidth'] = 4

In [2]:
from cav_casestudy.SUMO.spaceTimePlotWithSignals import  read_trajectory_xml, read_signal_xml, trajectory_process
from evaluation_utils.data_utils import get_data_dics
# data_dic_with_dynamics_before_calibration, data_dic_without_dynamics_before_calibration = get_data_dics('.\\Experiments\\BCV3')
data_dic_with_dynamics_after_calibration, data_dic_without_dynamics_after_calibration = get_data_dics('.\\Experiments_Sumo\\ACV3')
# test_trajectory_data = data_dic_with_dynamics_after_calibration[100.0]['path']
# test_trajectory_data = read_trajectory_xml(os.path.join(test_trajectory_data, 'fcd.xml'), refer_coord=[160, 735])

In [ ]:

import sumolib
from collections import deque

def get_lane_geometry(net_path):
    net: sumolib.net.Net = sumolib.net.readNet(net_path)
    lane_data = {}

    for edge in net.getEdges():
        edge: sumolib.net.edge.Edge = edge
        for lane in edge.getLanes():
            lane: sumolib.net.lane.Lane = lane
            lane_id = lane.getID()
            shape = lane.getShape()
            successors = [conn.getToLane().getID() for conn in lane.getOutgoing()]
            lane_data[lane_id] = {
                "edge_id": edge.getID(),
                # "speed": lane.getSpeed(),
                # "length": lane.getLength(),
                # "shape": shape,
                "successors": successors
            }

    # Build reverse links (predecessors)
    for lane_id in lane_data:
        lane_data[lane_id]["predecessors"] = []

    for lane_id, info in lane_data.items():
        for succ in info["successors"]:
            if succ in lane_data:
                lane_data[succ]["predecessors"].append(lane_id)
    return lane_data

def find_upstream_lanes(lane_data, start_lane_id, max_depth=3):
    visited = set()
    queue = deque([(start_lane_id, 0)])
    upstream_lanes = set()

    while queue:
        current_lane, depth = queue.popleft()
        if depth > max_depth or current_lane in visited:
            continue
        visited.add(current_lane)
        upstream_lanes.add(current_lane)
        predecessors = lane_data.get(current_lane, {}).get("predecessors", [])
        for pred in predecessors:
            queue.append((pred, depth + 1))
    return upstream_lanes

def get_upstream_veh_data_for_one_veh(row_data_ego, data_all_t, upsteam_vehcles_max_cnt=5, max_vehicles_distance=200):
    ego_x, ego_y = row_data_ego['x'], row_data_ego['y']
    up_stream_lanes = row_data_ego['up_stream_lanes']
    data_all_t = data_all_t[data_all_t['id'] != 'ego']
    candidates = data_all_t[data_all_t['lane'].isin(up_stream_lanes)].copy()
    if candidates.empty:
        return pd.DataFrame(columns=['speed', 'acceleration', 'distance'])
    candidates['distance'] = np.sqrt((candidates['x'] - ego_x) ** 2 + (candidates['y'] - ego_y) ** 2)
    candidates_sorted = candidates[candidates['distance'] <= max_vehicles_distance].sort_values(by='distance').head(upsteam_vehcles_max_cnt)

    return candidates_sorted[['id', 'speed', 'acceleration', 'distance']]


def preprocess_data(data_all, lane_geometry, upsteam_vehcles_max_cnt=5, source_veh_id='ego', step_length=0.1):
    relative_time = 28800
    wb_lanes = ['-2801', '-280', '-307', '-327', '-3271', '-281', '-315', '-3151', '-321', '-300', '-2851', '-285', '-290', '-298', '-295']
    eb_lanes = ['-312', '-293', '-297', '-288', '-2881', '-286', '-302', '-3221', '-322', '-313', '-284', '-2841', '-328', '-304']
    data_all['speed'] = data_all['speed'].astype(float)
    data_all['x'] = data_all['x'].astype(float)
    data_all['y'] = data_all['y'].astype(float)
    data_all['acceleration'] = data_all.groupby('id')['speed'].diff(periods=-1)/step_length
    data_all['acceleration'] = data_all['acceleration'].fillna(0.0)
    data_all = trajectory_process(data_all, eb_lanes, wb_lanes, relative_time)
    data_wb = data_all[data_all['direction'] == 'WB'].reset_index(drop=True)
    data_wb_ego = data_wb[data_wb['id'] == source_veh_id].copy()
    if data_wb_ego.empty:
        upstream_results_df = pd.DataFrame()
        return upstream_results_df
    # Compute upstream lanes once for each ego lane
    lane_to_upstream = {}
    # for lane in data_wb_ego['lane'].unique():
    #     lane_to_upstream[lane] = find_upstream_lanes(lane_geometry, lane)
    # data_wb_ego['up_stream_lanes'] = data_wb_ego['lane'].map(lane_to_upstream)
    data_wb_ego['up_stream_lanes'] = data_wb_ego['lane'].map(lambda lane: find_upstream_lanes(lane_geometry, lane))
    # Compute upstream vehicle data per row
    upstream_results = []
    for _, row in data_wb_ego.iterrows():
        t = row['time']
        data_all_t = data_all[data_all['time'] == t]
        upstream_veh_df = get_upstream_veh_data_for_one_veh(row, data_all_t, upsteam_vehcles_max_cnt)
        speeds = upstream_veh_df['speed'].tolist()
        accs = upstream_veh_df['acceleration'].tolist()
        # Pad if needed
        while len(speeds) < upsteam_vehcles_max_cnt:
            speeds.append(np.nan)
            accs.append(np.nan)
        result_row = {
            f"veh{i+1}_speed": speeds[i] for i in range(upsteam_vehcles_max_cnt)
        }
        result_row.update({
            f"veh{i+1}_acc": accs[i] for i in range(upsteam_vehcles_max_cnt)
        })
        result_row['time'] = t
        upstream_results.append(result_row)
    upstream_results_df = pd.DataFrame(upstream_results)
    
    ego_speed = data_wb_ego['speed'].values
    ego_acc = data_wb_ego['acceleration'].values
    upstream_results_df['ego_speed'] = ego_speed
    upstream_results_df['ego_acc'] = ego_acc
    
    return upstream_results_df

def get_upstream_veh_data(input_data_dic):
    all_results = {}
    
    for penetration_rate, data_dic in input_data_dic.items():
        print(penetration_rate)
        path = data_dic['path']
        # Load and preprocess
        lane_geometry = get_lane_geometry(os.path.join(path, 'chatt.net.xml'))
        refer_coord = [160, 735]
        data_all = read_trajectory_xml(os.path.join(path, 'fcd.xml'), refer_coord=refer_coord)

        upstream_results_df = preprocess_data(data_all, 
                                              lane_geometry,
                                              upsteam_vehcles_max_cnt=5,
                                              source_veh_id='ego',
                                              step_length=0.1)
        upstream_results_df.to_csv(os.path.join(path, f'upstream_results.csv'), index=False)
        # all_results[penetration_rate] = upstream_results_df
        
        
    return all_results

In [4]:
get_upstream_veh_data(data_dic_with_dynamics_after_calibration)
get_upstream_veh_data(data_dic_without_dynamics_after_calibration)

-1.0
0.0
10.0
20.0
30.0
40.0
50.0
60.0
70.0
80.0
90.0
100.0
-1.0
0.0
10.0
20.0
30.0
40.0
50.0
60.0
70.0
80.0
90.0
100.0


{}

In [ ]:
def plot_signal_changes_xml(ax, sumosignalresult, sumoSignalConfig, direction='EB', extra_label=""):
    
    sumoSignalConfig = sumoSignalConfig[sumoSignalConfig['approach_direction'] == direction].reset_index(drop=True)
    if direction == 'WB':
        sumoSignalConfig['distance'] = sumoSignalConfig['distance_wb']
    elif direction == 'EB':
        sumoSignalConfig['distance'] = sumoSignalConfig['distance_eb']
    for sc in sumoSignalConfig.id.unique():
        
        phase_num = sumoSignalConfig[sumoSignalConfig['id'] == sc]['name'].values[0]
        movement_indexes = eval(sumoSignalConfig[sumoSignalConfig['id'] == sc]['movement_index'].values[0])
        x_loc = sumoSignalConfig[sumoSignalConfig['id'] == sc]['distance'].values[0]

        signal_changes_tem = sumosignalresult[sumosignalresult['id'] == sc].reset_index(drop=True)
        signal_changes_tem['target_phase_state'] = signal_changes_tem['state'].str[movement_indexes[0]: movement_indexes[0] + 1]

        # only keep phase changes for the targeted phase
        signal_changes_tem = signal_changes_tem[signal_changes_tem['target_phase_state'] != signal_changes_tem['target_phase_state'].shift()]
        signal_changes_tem['target_phase_state'] = signal_changes_tem['target_phase_state'].map({'G': 'green', 'g': 'green', 'y': 'yellow', 'r': 'red', 's': 'red'})
        
        cnt = 1
        prev_idx = 0
        prev_clr = signal_changes_tem.loc[prev_idx,'target_phase_state']
        
        for index, row in signal_changes_tem.iterrows():
            if index >= 1:
                
                y1 = [x_loc, x_loc]
                x1 = [float(signal_changes_tem.loc[prev_idx,'time']), float(row.time)]
                ax.plot(x1, y1, color=prev_clr, linewidth=7, solid_capstyle='butt')
                c = row.target_phase_state

                prev_idx = index
                prev_clr = c
                cnt += 1
            else:
                continue

    ax.set_title(f'Shallowford Rd Space-Time Diagram {direction} {extra_label}')
    ax.set_xlabel('Simulation Second (s)', fontsize=20)
    ax.set_ylabel(f'Distance{direction}', fontsize=20)
    # ax.set_xlim(plot_start_time, plot_start_time + duration_limit)
    ax.set_yticks(sumoSignalConfig['distance'].values)
    ax.set_yticklabels(sumoSignalConfig['id'].values, fontsize=20)
    # ax.invert_yaxis()
    return ax
def plot_trajectory(ax, trajectory_data, highlight_veh_number='None', plot_start_time=29000, plot_end_time=32400):
    line_width = 1
    trajectory_data['time'] = trajectory_data['time'].astype(float)
    trajectory_data['distance'] = trajectory_data['distance'].astype(float)
    highlight_veh_data = trajectory_data[trajectory_data['id'] == highlight_veh_number]
    trajectory_data = trajectory_data[trajectory_data['id'] != highlight_veh_number]


    sns.lineplot(ax=ax, data=trajectory_data[trajectory_data['type'] == 'HDV'], x='time', y='distance', hue='id', legend=False, palette=['lightgrey'])
    sns.lineplot(ax=ax, data=trajectory_data[trajectory_data['type'] == 'CAV'], x='time', y='distance', hue='id', legend=False, palette=['grey'])
    
    highlight_veh_data = assign_trip_id(highlight_veh_data, 100)
    # highlight a specific vehicle
    if highlight_veh_number != 'None':
        
        clr = 'lightgreen'
        line_width = 4
        for group_name, group in highlight_veh_data.groupby('tripId'):
            ax.plot(group['time'].astype(float), group['distance'].astype(float), color=clr, linewidth=line_width, solid_capstyle='butt')
    ax.set_xlim(0, 3600)
    return ax


from cav_casestudy.SUMO.spaceTimePlotWithSignals import  read_trajectory_xml, read_signal_xml, trajectory_process
# import sns
import seaborn as sns
def plot_space_time_diagram_calibration(data_dic_before_calibration, data_dic_after_calibration):
    refer_coord_wb = [1380, 225]
    refer_coord_eb = [200, 709]
    setting_cnt = len(data_dic_before_calibration)
    fig, axes = plt.subplots(setting_cnt, 2, figsize=(40, 5 * setting_cnt))
    # plot the west bound first
    for idx, pr in enumerate(data_dic_before_calibration.keys()):
        path_before_calibration = data_dic_before_calibration[pr]['path']
        sumo_signal_before_calibration = read_signal_xml(os.path.join(path_before_calibration, 'signal_result.xml'))
        sumoSignalConfig_before_calibration = data_dic_before_calibration[pr]['sumo_signal_config']
        trajectory_data_wb_before_calibration = read_trajectory_xml(os.path.join(path_before_calibration, 'fcd.xml'), refer_coord=refer_coord_wb)

        path_after_calibration = data_dic_after_calibration[pr]['path']
        sumo_signal_after_calibration = read_signal_xml(os.path.join(path_after_calibration, 'signal_result.xml'))
        sumoSignalConfig_after_calibration = data_dic_after_calibration[pr]['sumo_signal_config']
        trajectory_data_wb_after_calibration = read_trajectory_xml(os.path.join(path_after_calibration, 'fcd.xml'), refer_coord=refer_coord_wb)

        wb_lanes = ['-2801', '-280', '-307', '-327', '-3271', '-281', '-315', '-3151', '-321', '-300', '-2851', '-285', '-290', '-298', '-295']
        eb_lanes = ['-312', '-293', '-297', '-288', '-2881', '-286', '-302', '-3221', '-322', '-313', '-284', '-2841', '-328', '-304']
        relative_time = 28800
        sumo_signal_before_calibration['time'] = sumo_signal_before_calibration['time'].astype(float) - relative_time
        sumo_signal_after_calibration['time'] = sumo_signal_after_calibration['time'].astype(float) - relative_time

        
        
        
        trajectory_data_wb_before_calibration = trajectory_process(trajectory_data_wb_before_calibration, eb_lanes, wb_lanes, relative_time)
        trajectory_data_wb_after_calibration = trajectory_process(trajectory_data_wb_after_calibration, eb_lanes, wb_lanes, relative_time)

        trajectory_data_wb_after_calibration = trajectory_data_wb_after_calibration[trajectory_data_wb_after_calibration['direction'] == 'WB'].reset_index(drop=True)
        trajectory_data_wb_before_calibration = trajectory_data_wb_before_calibration[trajectory_data_wb_before_calibration['direction'] == 'WB'].reset_index(drop=True)


        axes[idx][0] = plot_trajectory(axes[idx][0], trajectory_data_wb_before_calibration, highlight_veh_number='ego')
        axes[idx][0] = plot_signal_changes_xml(axes[idx][0], sumo_signal_before_calibration, sumoSignalConfig_before_calibration, direction='WB', extra_label=f'PR: {pr:.0f}% Before Calibration')

        
        axes[idx][1]  = plot_trajectory(axes[idx][1], trajectory_data_wb_after_calibration, highlight_veh_number='ego')
        axes[idx][1] = plot_signal_changes_xml(axes[idx][1], sumo_signal_after_calibration, sumoSignalConfig_after_calibration, direction='WB', extra_label=f'PR: {pr:.0f}% After Calibration')
    plt.tight_layout()
    plt.show()
    
def plot_space_time_diagram(data_dic, name):
    setting_cnt = len(data_dic)
    fig, axes = plt.subplots(setting_cnt, 2, figsize=(40, 5 * setting_cnt))
    for idx, pr in enumerate(data_dic.keys()):
        sumo_signal = data_dic[pr]['signal_file']
        sumoSignalConfig = data_dic[pr]['sumo_signal_config']
        trajectory_data_wb = data_dic[pr]['trajectory_data_wb']
        trajectory_data_eb = data_dic[pr]['trajectory_data_eb']
        label = f'PR: {pr:.0f} {name}%'
        
        axes[idx][0] = plot_trajectory(axes[idx][0], trajectory_data_wb, 100, highlight_veh_number='ego')
        axes[idx][0] = plot_signal_changes_xml(axes[idx][0], sumo_signal, sumoSignalConfig, direction='WB', extra_label=label)

        
        axes[idx][1]  = plot_trajectory(axes[idx][1], trajectory_data_eb, 100, highlight_veh_number='ego')
        axes[idx][1] = plot_signal_changes_xml(axes[idx][1], sumo_signal, sumoSignalConfig, direction='EB', extra_label=label)

    plt.tight_layout()
    
def plot_profiles(data_dic, unit_dic, name):
    # drop the last trip
    # eco_profile = eco_profile[eco_profile['Trip_ID'] != eco_profile['Trip_ID'].iloc[-1]]
    # baseline_profile = baseline_profile[baseline_profile['Trip_ID'] != baseline_profile['Trip_ID'].iloc[-1]]
    data_fileds_num = len(unit_dic)
    fig, axs = plt.subplots(data_fileds_num, 1, figsize=(40, 5 * data_fileds_num))
    # set the title for the plot
    fig.suptitle(name, fontsize=20)
    for i, (field, unit) in enumerate(unit_dic.items()):
        # plot the data for each direction with different color
        # for group_name, group in eco_profile.groupby('Trip_ID'):
        #     axs[i].plot(group['Time'], group[field], label='EcoDriving')
        for pr in data_dic.keys():
            eco_profile = data_dic[pr]['ego_profile']
            pr = f"Penetration Rate: {pr:.0f} %"
            axs[i].plot(eco_profile['Time'], eco_profile[field], label=pr)
            axs[i].set_ylabel(unit)
            axs[i].set_xlabel('Time (s)')
            axs[i].legend()
            axs[i].grid()
    plt.tight_layout()
    # plt.savefig(f'{name}.png')



def plot_sub_profiles(eco_profile, eco_trip_id, baseline_profile, baseline_trip_id, unit_dic, name, direction='WB'):
    eco_profile = eco_profile[eco_profile['Trip_ID'] == eco_trip_id]
    baseline_profile = baseline_profile[baseline_profile['Trip_ID'] == baseline_trip_id]
    eco_profile = eco_profile[eco_profile['Direction'] == direction]
    baseline_profile = baseline_profile[baseline_profile['Direction'] == direction]
    
    data_fileds_num = len(unit_dic)
    fig, axs = plt.subplots(data_fileds_num, 1, figsize=(40, 5 * data_fileds_num))
    for i, (field, unit) in enumerate(unit_dic.items()):
        axs[i].plot(eco_profile['Time'], eco_profile[field], label='EcoDriving')
        axs[i].plot(baseline_profile['Time'], baseline_profile[field], label='Baseline')
        axs[i].set_ylabel(unit)
        axs[i].set_xlabel('Time (s)')
        axs[i].legend()
        axs[i].grid()
    plt.tight_layout()
    # plt.savefig(f'{name}.png')


In [ ]:
ev_unit_dic = {'DesiredSpeed': 'Desired Speed (m/s)', 
                'ActualSpeed' : 'Actual Speed (m/s)', 
                'MPGe' : 'MPGe',
                'BatteryPower': 'Battery Power (W)', 
                'MotorSpeed': 'Motor Speed (rad/s)', 
                'MotorTorque': 'Motor Torque (Nm)', 
                'BatterySOC': 'State of Charge (%)', 
                'BatteryCurrent': 'Current (A)',
                # 'Distance': 'Distance (m)',
                # 'EnergyConsumption': 'Energy Consumption (KWH)',
                }

icv_unit_dic = {'DesiredSpeed' : 'Desired Speed (m/s)',
                'ActualSpeed' : 'Actual Speed (m/s)',
                'MPG': 'MPG', 
                'FuelConsumptionRate': 'Fuel Rate (m^3/s)',
                'EngineSpeed' : 'Engine Speed (rad/s)', 
                'EngineTorque' : 'Engine Torque (Nm)',
                # 'FuelConsumption': 'Fuel Consumption (m^3)',
                # 'Distance': 'Distance (m)',
                }

# plot_sub_profiles(eco_icv_profile, 3, baseline_icv_profile, 3, icv_unit_dic, 'ICV_Trip_1', direction='WB')
# plot_profiles(data_dic=data_dic_with_dynamics, unit_dic=ev_unit_dic, name='EV')
# plot_profiles(data_dic=data_dic_without_dynamics, unit_dic=ev_unit_dic, name='Baseline')
# plot_profiles(data_dic=data_dic_without_dynamics, unit_dic=ev_unit_dic, name='Baseline')
# plot_space_time_diagram(data_dic_with_dynamics, 'With Dynamics')
# plot_space_time_diagram_calibration(data_dic_with_dynamics_before_calibration, data_dic_with_dynamics_after_calibration)
plot_space_time_diagram_calibration(data_dic_with_dynamics_before_calibration, data_dic_with_dynamics_after_calibration)
plot_space_time_diagram_dynamics(data_dic_with_dynamics_after_calibration, data_dic_without_dynamics_after_calibration)

# plot_profiles(eco_icv_profile, baseline_icv_profile, icv_unit_dic, 'ICV')
# plot_profiles(eco_icv_profile, baseline_icv_profile, ev_unit_dic, 'ICV')
# plot_profiles(eco_ev_profile, baseline_ev_profile, ev_unit_dic, 'EV')

In [ ]:
#data_dic_with_dynamics_before_calibration data_dic_with_dynamics_after_calibration


penetration_rates_to_plot = [0.0, 50.0, 100.0]
trip_max_limit = 4
fig, axes = plt.subplots(len(penetration_rates_to_plot), 2, figsize=(24, 12))


speed_list_total_before_calibration = []
time_list_total_before_calibration = []

speed_list_total_after_calibration = []
time_list_total_after_calibration = []

for idx, penetration_rate in enumerate(penetration_rates_to_plot):
    data_dic_before_calibration = data_dic_with_dynamics_before_calibration[penetration_rate]['trajectory_data_wb']
    data_dic_after_calibration = data_dic_with_dynamics_after_calibration[penetration_rate]['trajectory_data_wb']
    data_dic_before_calibration = data_dic_before_calibration[data_dic_before_calibration['id'] == 'ego']
    data_dic_after_calibration = data_dic_after_calibration[data_dic_after_calibration['id'] == 'ego']
    data_dic_before_calibration = assign_trip_id(data_dic_before_calibration, 100)
    data_dic_after_calibration = assign_trip_id(data_dic_after_calibration, 100)
    
    if len(data_dic_before_calibration['tripId'].unique()) < trip_max_limit:
        trip_limit = len(data_dic_before_calibration['tripId'].unique())
    else:
        trip_limit = trip_max_limit

    if len(data_dic_after_calibration['tripId'].unique()) < trip_max_limit:
        trip_limit = len(data_dic_after_calibration['tripId'].unique())
    else:
        trip_limit = trip_max_limit

    # uniformly pick trip_limit trips, not the first trips, for examople, if there are 10 trips, we pick 2, 4, 6, 8, 10
    
    trip_id_list = data_dic_before_calibration['tripId'].unique()
    trip_id_list = trip_id_list[1::2]
    trip_id_list = trip_id_list[:trip_limit]
    data_dic_before_calibration = data_dic_before_calibration[data_dic_before_calibration['tripId'].isin(trip_id_list)]

    trip_id_list = data_dic_after_calibration['tripId'].unique()
    trip_id_list = trip_id_list[1::2]
    trip_id_list = trip_id_list[:trip_limit]
    data_dic_after_calibration = data_dic_after_calibration[data_dic_after_calibration['tripId'].isin(trip_id_list)]

    speed_list_before_calibration = []
    time_list_before_calibration = []
    for trip_id, group in data_dic_before_calibration.groupby('tripId'):
        speed = group['speed'].astype(float)
        time = group['time'].astype(float)
        
        time_shifted = time - time.iloc[0]
        speed_list_before_calibration.append(speed.values.tolist())
        time_list_before_calibration.append(time_shifted.values.tolist())
    speed_list_total_before_calibration.append(speed_list_before_calibration)
    time_list_total_before_calibration.append(time_list_before_calibration)

    speed_list_after_calibration = []
    time_list_after_calibration = []
    for trip_id, group in data_dic_after_calibration.groupby('tripId'):
        speed = group['speed'].astype(float)
        time = group['time'].astype(float)
        
        time_shifted = time - time.iloc[0]
        speed_list_after_calibration.append(speed.values.tolist())
        time_list_after_calibration.append(time_shifted.values.tolist())
    speed_list_total_after_calibration.append(speed_list_after_calibration)
    time_list_total_after_calibration.append(time_list_after_calibration)


speed_max_limit_before_calibration = max([max(max(speed_list)) for speed_list in speed_list_total_before_calibration])
time_max_limit_before_calibration = max([max(max(time_list)) for time_list in time_list_total_before_calibration])

speed_max_limit_after_calibration = max([max(max(speed_list)) for speed_list in speed_list_total_after_calibration])
time_max_limit_after_calibration = max([max(max(time_list)) for time_list in time_list_total_after_calibration])

speed_max_limit = max(speed_max_limit_before_calibration, speed_max_limit_after_calibration) * 1.4
time_max_limit = max(time_max_limit_before_calibration, time_max_limit_after_calibration)





for idx, penetration_rate in enumerate(penetration_rates_to_plot):
    for i in range(len(speed_list_total_before_calibration[idx])):
        axes[idx][0].plot(time_list_total_before_calibration[idx][i], speed_list_total_before_calibration[idx][i], label=f'')
        
    for i in range(len(speed_list_total_after_calibration[idx])):
        axes[idx][1].plot(time_list_total_after_calibration[idx][i], speed_list_total_after_calibration[idx][i], label=f'')

    axes[idx][0].set_title(f'Ego Vehicle Speed at {penetration_rate} % Before Calibration (With Dynamics)', fontsize=14)
    axes[idx][0].set_title(f'Ego Vehicle Speed at {penetration_rate} % Before Calibration (With Dynamics)', fontsize=14)
    axes[idx][0].set_xlabel('Time (s)', fontsize=12)
    axes[idx][0].set_ylabel('Speed (m/s)', fontsize=12)
    axes[idx][0].set_ylim(0, speed_max_limit)
    axes[idx][0].set_xlim(0, time_max_limit)
    axes[idx][0].grid()

    axes[idx][1].set_title(f'Ego Vehicle Speed at {penetration_rate} % After Calibration (With Dynamics)', fontsize=14)
    axes[idx][1].set_title(f'Ego Vehicle Speed at {penetration_rate} % After Calibration (With Dynamics)', fontsize=14)
    axes[idx][1].set_xlabel('Time (s)', fontsize=12)
    axes[idx][1].set_ylabel('Speed (m/s)', fontsize=12)
    axes[idx][1].set_ylim(0, speed_max_limit)
    axes[idx][1].set_xlim(0, time_max_limit)
    axes[idx][1].grid()

plt.tight_layout()

In [ ]:
print('Before Calibration (WB)')
for penetration_rate, data_dic in data_dic_with_dynamics_before_calibration.items():
    veh_cnt = data_dic['trajectory_data_wb']['id'].nunique()
    print(f'Penetration Rate: {penetration_rate} %, Vehicle Count: {veh_cnt}')
print('After Calibration (WB)')

for penetration_rate, data_dic in data_dic_with_dynamics_after_calibration.items():
    veh_cnt = data_dic['trajectory_data_wb']['id'].nunique()
    print(f'Penetration Rate: {penetration_rate} %, Vehicle Count: {veh_cnt}')

In [ ]:
def get_corridor_veh_cnt(trajectory_data_eco_driving):
    
    trajectory_data_eco_driving = trajectory_data_eco_driving[trajectory_data_eco_driving['time'].astype(float) >= 29000].reset_index(drop=True)
    # only keep the major corridor trajectories
    wb_lanes = ['-2801', '-280', '-307', '-327', '-3271', '-281', '-315', '-3151', '-321', '-300', '-2851', '-285', '-290', '-298', '-295']
    eb_lanes = ['-312', '-293', '-297', '-288', '-2881', '-286', '-302', '-3221', '-322', '-313', '-284', '-2841', '-328', '-304']
    trajectory_data_eco_driving['segment'] = trajectory_data_eco_driving['lane'].str.split('_').str[0]
    trajectory_data_eco_driving['direction'] = np.where(trajectory_data_eco_driving['segment'].isin(wb_lanes), "WB",
                                            np.where(trajectory_data_eco_driving['segment'].isin(eb_lanes), "EB", None))
    trajectory_data_eco_driving = trajectory_data_eco_driving.sort_values(by=['id', 'time'], ignore_index=True)
    trajectory_data_eco_driving = trajectory_data_eco_driving[(trajectory_data_eco_driving['direction'].notnull())
                                                              | (trajectory_data_eco_driving['segment'].str.contains(':'))]
    # Forward fill and backward fill within each group
    trajectory_data_eco_driving = remove_none_edges(trajectory_data_eco_driving, group_col="id", value_col="direction")
    # Forward fill and backward fill within each group
    trajectory_data_eco_driving["direction"] = trajectory_data_eco_driving.groupby("id")["direction"].ffill()
    trajectory_data_eco_driving = trajectory_data_eco_driving[trajectory_data_eco_driving['direction'].isin(['EB', 'WB'])].reset_index(drop=True)

    return trajectory_data_eco_driving['id'].nunique()
print('Before Calibration (WB)')
for penetration_rate, data_dic in data_dic_with_dynamics_before_calibration.items():
    veh_cnt = get_corridor_veh_cnt(data_dic['trajectory_data_wb'])
    print(f'Penetration Rate: {penetration_rate} %, Vehicle Count: {veh_cnt}')
print('After Calibration (WB)')

for penetration_rate, data_dic in data_dic_with_dynamics_after_calibration.items():
    veh_cnt = get_corridor_veh_cnt(data_dic['trajectory_data_wb'])
    print(f'Penetration Rate: {penetration_rate} %, Vehicle Count: {veh_cnt}')

In [ ]:
print('Before Calibration (WB)')
vtMicroCoeff = pd.read_csv(r'VTMicroCoeff.csv')
df_vehicle_src_coeff = pd.read_csv(r'VehicleSrcCoeff.csv')
for penetration_rate, data_dic in data_dic_with_dynamics_before_calibration.items():
    (trajectory_data_energy_sum, system_fuel_consume_vt_cpmf_liter, system_fuel_consume_vt_micro_liter, system_energy_consume_tractive_kj,
            system_energy_consume_tractive_regen_kj, system_travel_dist_m, system_travel_dist2_m, system_travel_time_s) = get_traj_eval(data_dic['trajectory_data_wb'], 1, vtMicroCoeff, df_vehicle_src_coeff, veh_no='system')
    print(f'Penetration Rate: {penetration_rate} %, Mean Speed: {system_travel_dist2_m / system_travel_time_s * 2.23694} mph')
print('After Calibration (WB)')

for penetration_rate, data_dic in data_dic_with_dynamics_after_calibration.items():
    (trajectory_data_energy_sum, system_fuel_consume_vt_cpmf_liter, system_fuel_consume_vt_micro_liter, system_energy_consume_tractive_kj,
            system_energy_consume_tractive_regen_kj, system_travel_dist_m, system_travel_dist2_m, system_travel_time_s) = get_traj_eval(data_dic['trajectory_data_wb'], 1, vtMicroCoeff, df_vehicle_src_coeff, veh_no='system')
    print(f'Penetration Rate: {penetration_rate} %, Mean Speed: {system_travel_dist2_m / system_travel_time_s * 2.23694} mph')